# Bronze ingestion test

Ingests `data/B001/customer_B001.csv` into a bronze table using the `pipeline` functions directly. The cells show:
- a first load (all rows inserted)
- a rerun of the same file (idempotency: every row is a replay, nothing inserted)
- a failed run, with the error naming the step

In [10]:
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from pyspark.sql.types import StructType, StructField, StringType
from pipeline.session import get_spark_session
from pipeline.catalog import ensure_schema_exists, get_schema_fqn, get_table_fqn
from pipeline.audit import AuditRun
from pipeline.errors import step
from pipeline.metadata import add_metadata
from pipeline.reader import read_source_csv
from pipeline.validation import assert_columns_exist, count_duplicate_keys, count_null_business_keys
from pipeline.writer import insert_new_rows
from bronze.entity import EntityConfig

spark = get_spark_session('bronze_ingestion_test')

In [11]:
COLUMNS = [
    'source_record_id', 'customer_id', 'first_name', 'last_name', 'email', 'phone',
    'date_of_birth', 'loyalty_status', 'marketing_opt_in', 'created_at', 'updated_at',
]
# Bronze keeps every source column as a string
SCHEMA = StructType([StructField(c, StringType(), True) for c in COLUMNS])

PROJECT_DATA = root / 'data' / 'B001'
CUSTOMER_FILE = PROJECT_DATA / 'customer_B001.csv'

CATALOG = 'forge'
TEST_SCHEMA = 'bronze_ingestion_test'
ensure_schema_exists(spark, CATALOG, TEST_SCHEMA)

entity = EntityConfig(
    name='customer',
    source_path=str(CUSTOMER_FILE),
    schema=SCHEMA,
    target_table=get_table_fqn(CATALOG, TEST_SCHEMA, 'customer'),
    merge_keys=('customer_id',),
)

print('source:', entity.source_path)
print('target:', entity.target_table)

source: /home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/customer_B001.csv
target: bronze_ingestion_test.customer


In [12]:
records = []  # the audit sink: AuditRun appends each finished AuditRecord here

def ingest(path: str):
    with AuditRun('bronze', entity.name, entity.target_table, sink=records.append) as run:
        with step('read', path=path):
            df = read_source_csv(spark, entity.schema, path, entity.read_options, file_path_column='_source_file')

        with step('add_metadata'):
            df = add_metadata(
                df,
                run_id=run.record.run_id,
                ingested_at=datetime.now(timezone.utc),
                source_columns=[f.name for f in entity.schema.fields],
            )

        with step('validate'):
            assert_columns_exist(df, entity.merge_keys)
            run.add('source_row_count', df.count())
            run.add('null_business_key_count', count_null_business_keys(df, entity.merge_keys))
            run.add('duplicate_business_key_count', count_duplicate_keys(df, entity.merge_keys))

        with step('write', table=entity.target_table):
            result = insert_new_rows(spark, df, entity.target_table, entity.merge_keys, hash_column='_record_hash')
            for name, value in asdict(result).items():
                run.add(name, value)

    return records[-1]

def show(record):
    print(record.status, record.metrics, record.error_message or '')

In [13]:
# First load: every row should be inserted
show(ingest(entity.source_path))

success {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 0, 'inserted_row_count': 50, 'hash_conflict_count': 0, 'target_total_row_count': 50} 


In [14]:
# Rerun of the same file: every row is a replay, so nothing is inserted
show(ingest(entity.source_path))

success {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 50, 'inserted_row_count': 0, 'hash_conflict_count': 0, 'target_total_row_count': 50} 


In [15]:
# Table contents: one row per customer_id
spark.read.table(entity.target_table).orderBy('customer_id').show(truncate=False)

+---------------------------+-----------+----------+----------+----------------------------------------+--------------+-------------+--------------+----------------+-------------------+-------------------+------------------------------------------------------------------------------------------+----------------------------------------------------------------+------------------------------------+--------------------------+
|source_record_id           |customer_id|first_name|last_name |email                                   |phone         |date_of_birth|loyalty_status|marketing_opt_in|created_at         |updated_at         |_source_file                                                                              |_record_hash                                                    |_run_id                             |_ingested_at              |
+---------------------------+-----------+----------+----------+----------------------------------------+--------------+-------------+--------------+

In [16]:
# A missing source file: expect status 'failed' and an error naming the 'read' step
try:
    ingest(str(PROJECT_DATA / 'missing.csv'))
except Exception as exc:
    print('raised:', exc)

show(records[-1])

raised: Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03
failed {} Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03


In [17]:
spark.sql(f'DROP SCHEMA IF EXISTS {get_schema_fqn(CATALOG, TEST_SCHEMA)} CASCADE')

DataFrame[]